<a href="https://colab.research.google.com/github/mtchka/tsukuten_caption_maker/blob/main/tsukuten_caption_maker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# つくてんキャプションメーカー
筑波大学天文研究会の、雙峰祭での写真展などで制作するキャプションのデータをつくるプログラムです。

## (1) ライブラリのインストール
パワポをいじるための拡張機能みたいなやつをインストールします。
以下のコマンドを実行してください。

In [1]:
!pip install python-pptx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.6 MB/s eta 0:00:00


In [2]:
from google.colab import files

# ファイルをアップロードする
uploaded = files.upload()

filename = list(uploaded.keys())[0]

Saving 2025年度雙峰祭　天文写真展　募集（回答） - フォームの回答 1.csv to 2025年度雙峰祭　天文写真展　募集（回答） - フォームの回答 1.csv


In [4]:
from pptx import Presentation
from pptx.util import Pt, Cm
from pptx.enum.text import PP_ALIGN
from pptx.enum.shapes import MSO_CONNECTOR
from pptx.dml.color import RGBColor
import csv


slide_width =  Cm(29.7)
slide_height = Cm(21.0)
# フォントは無料のZen Maru Gothic（プログラム実行前にPCにダウンロードが必要）
font = 'Zen Maru Gothic'


# タイトルと説明の間に線を引くだけの関数
def line_maker(slide):
    line = slide.shapes.add_connector(
        MSO_CONNECTOR.STRAIGHT,
        Cm(2), Cm(6), slide_width-Cm(2), Cm(6)
    )
    line.line.color.rgb = RGBColor(0, 0, 0)
    line.shadow.inherit = False

# テキストを配置する関数
# 引数は(スライド、テキストボックスのx座標, y座標, 幅, 高さ, 文字l(左)c(中央)r(右)寄せ, 文字折り返し(True or False), 文字サイズ, テキスト内容)
# ↑引数なんかもっとうまくできるだろ
def text_maker(slide, left, top, width, alignment, wrap, pt, text):
    textbox = slide.shapes.add_textbox(left, top, width, Cm(10))
    frame = textbox.text_frame
    p = frame.paragraphs[0]
    run = p.add_run()
    run.font.name = font
    run.font.size = Pt(pt)
    run.text = text
    frame.word_wrap = wrap

    if alignment == 'c':
        p.alignment = PP_ALIGN.CENTER
    elif alignment == 'r':
        p.alignment = PP_ALIGN.RIGHT

# なんかいい感じに文字配置する関数（力技だよ）
# 引数はキャプション情報1セット
def caption_maker(slide, caption):
    text_title = f'{caption[2]}'
    text_maker(slide, (slide_width-Cm(10))/2, Cm(2), Cm(10), 'c', False, 66, text_title)

    text_camera = f'\n\n\nカメラ：'
    if caption[8] != '':
        text_camera += '\n鏡筒：'
    text_maker(slide, Cm(2.5), Cm(7.5), Cm(4), 'r', False, 26, text_camera)

    photo_info = ''
    if caption[9] != '':
        info_9 = f'露光{caption[9]}s'
        photo_info += f'{info_9:<10}'
    if caption[10] != '':
        info_10 = f'ISO{caption[10]}'
        photo_info += f'{info_10:<11}'
    if caption[11] != '':
        info_11 = f'f{caption[11]}'
        photo_info += f'{info_11:<14}'
    if caption[12] != '':
        photo_info += f'{caption[12]}枚スタック'

    text_contents = f'{caption[5]} {caption[6]}\n{caption[4]}\n\n\
{caption[7]}\n{caption[8]}\n{photo_info}\n\n{caption[3]}\n'
    text_maker(slide, Cm(6),Cm(7.5), slide_width-Cm(9), 'l', True, 26, text_contents)

    text_name = f'{caption[1]}　{caption[0]}'
    text_maker(slide, slide_width-Cm(12), slide_height-Cm(2.5), Cm(10), 'r', False, 26, text_name)


# 白紙パワポを作成➡キャプションデータ読み込み
# ➡キャプションごとにページを追加し、そこにキャプション情報を配置
# ➡全キャプションできたらパワポとして保存 という流れのmain関数
def main():
    prs = Presentation()
    prs.slide_width = slide_width
    prs.slide_height = slide_height

    captions = []
    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        # 各rowの要素は以下の通り。フォーム変えるならここを変える必要が出てくるかも
        # [名前, 期, タイトル, 説明, 場所, 日付, 時間, カメラ, 鏡筒, 露光時間, ISO, f値, スタック枚数]
        # これももっとうまくできるだろって感じだけどまあいいか
        for row in reader:
            captions.append(row)

    for caption in captions:
        slide = prs.slides.add_slide(prs.slide_layouts[6])
        line_maker(slide)
        caption_maker(slide, caption)

    prs.save("captions.pptx")
    files.download("captions.pptx")


if __name__ == '__main__':
    main()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>